# UKRI analysis

In [86]:
import pandas as pd
from discovery_child_development import PROJECT_DIR

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

In [ ]:
# AltairSaver = altair_save_utils.AltairSaver()

In [87]:
import utils
from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import importlib
importlib.reload(utils);

## Load data

In [100]:
# Gateway to Research labelled data
importlib.reload(utils);
data_df = utils.load_crunchbase_data().query("topics != 'arts'")

In [89]:
# Taxonomy dataframe
topics_df = utils.load_topic_data()

In [101]:
# Transform to one id and topic pair per row
importlib.reload(utils)
data_exploded_df = utils.explode_data(data_df).query("topics != 'arts'")

## Baseline trends

Baseline UKRI trends for funding and project counts 

In [102]:
importlib.reload(utils)
baseline_df = utils.get_baseline_crunchbase()

In [103]:
baseline_df

,year,counts,amount
0,2013,21249,4.235161e+07
1,2014,28412,6.857297e+07
2,2015,34201,1.044885e+08
3,2016,35601,1.208383e+08
4,2017,37020,1.541349e+08
5,2018,41433,2.166157e+08
6,2019,41130,1.975202e+08
7,2020,41158,2.229402e+08
8,2021,51862,4.485599e+08
9,2022,46549,3.684496e+08


In [104]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = baseline_df,
    year_start = 2019,
    year_end = 2023  
)
trends_baseline

,magnitude,growth
counts,4.327080e+04,12.111253
amount,2.906603e+08,81.751636


In [105]:
fig = pu.ts_smooth(
    baseline_df.assign(Total="Total").assign(amount = lambda df: df.amount/1000),
    ["Total"],
    variable= "amount",
    variable_title = "Total funding (£ millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

## Insight 0: Overall trends

Early-years project growth of funding and project counts trends



In [106]:
ts_counts = utils.get_timeseries(data_df, column='id')
ts_amounts = utils.get_timeseries(data_df, column='amount')

In [107]:
ts_amounts

,year,amount
0,2013,2.376831e+05
1,2014,5.037585e+05
2,2015,1.017056e+06
3,2016,1.391808e+06
4,2017,7.990981e+05
5,2018,9.816925e+05
6,2019,1.142650e+06
7,2020,1.021373e+06
8,2021,3.404494e+06
9,2022,1.067520e+06


In [108]:
au.ts_magnitude_growth_(
    ts_df = ts_counts,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
counts,322.8,-31.142153


In [109]:
au.ts_magnitude_growth_(
    ts_df = ts_amounts,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
amount,1.460740e+06,75.809079


In [110]:
fig = pu.ts_smooth(
    ts_amounts.assign(Total="Total").assign(amount = lambda df: df.amount/1000),
    ["Total"],
    variable= "amount",
    variable_title = "Total funding (£ millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

In [111]:
utils.get_data_distribution(data_exploded_df, column='type', values=['id', 'amount'])

,type,counts,counts_prop,amount,amount_prop
0,Biosciences,60,0.016,161972.363694,0.013
1,Child care & preschool,458,0.122,963432.277972,0.079
2,Development & learning,724,0.193,2702590.108185,0.221
3,General,1494,0.399,5237881.434438,0.428
4,Health,1222,0.326,4262038.884892,0.348
5,Parenting,655,0.175,1446843.706877,0.118
6,Social,442,0.118,1136190.561761,0.093
7,Technology,1579,0.421,4266466.106159,0.349


In [112]:
importlib.reload(utils)
ts_df = (
    utils.get_data_distribution(data_exploded_df, column='type', values=['id', 'amount'], ts=True)
    .query("type != 'General'")
)
utils.get_data_magnitude_growth(data_exploded_df, ids=None, column='type', value='amount')

,magnitude,growth,type,counts
2,397212.444321,185.800150,Development & learning,315
4,598976.843532,168.085441,Health,634
3,566310.503122,107.864211,General,664
6,158979.503472,84.343152,Social,177
5,155090.392865,62.644357,Parenting,268
7,494929.710705,44.777394,Technology,600
1,113498.561830,17.456791,Child care & preschool,188
0,13682.748113,-25.268116,Biosciences,35


In [113]:
fig = pu.ts_smooth(
    ts_df,
    ts_df['type'].unique(),
    variable= "amount",
    variable_title = "",
    category_column = 'type',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 1: Technology trends

- Magnitude and growth for technology topic overall
- Distribution of different technologies
- Growth of different technologies in UKRI funding


### Overall technology topic growth

In [114]:
tech_subtypes = set(topics_df.query("type == 'Technology'").subtype.unique())
tech_subtypes

{'AI', 'Immersive tech', 'Internet', 'Mobile'}

In [115]:
tech_type_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'type'])
)

In [116]:
ts_amounts_tech = utils.get_timeseries(tech_type_df, column='amount')
ts_counts_tech = utils.get_timeseries(tech_type_df, column='id')
utils.plot_quick_ts(ts_amounts_tech, 'amount')

alt.Chart(...)

In [117]:
au.ts_magnitude_growth_(ts_amounts_tech, year_start = 2019, year_end = 2023)

,magnitude,growth
amount,494929.710705,44.777394


In [142]:
494929/1000

494.929

### Distribution of different technologies

In [118]:
tech_subtype_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    .query("type == 'Technology'")
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'subtype'])
)

In [119]:
# Total tech funding
amount_total = tech_subtype_df.drop_duplicates('id').query("year >= 2019").amount.sum()

In [120]:
tech_subtype_dist = (
    tech_subtype_df
    .query("year >= 2019")
    .groupby('subtype')
    .agg(
        counts=('id', 'nunique'), 
        amount=('amount', 'sum')
    )
    .reset_index()
    .assign(amount_prop = lambda df: round(df.amount / amount_total, 3))
)

tech_subtype_dist

,subtype,counts,amount,amount_prop
0,AI,264,7.235923e+05,0.292
1,Immersive tech,118,4.137965e+05,0.167
2,Internet,94,6.209938e+05,0.251
3,Mobile,302,1.631140e+06,0.659


### Growth of technology topics

In [121]:
column = 'subtype'
value = 'amount'

tech_subtype_ts = (
    tech_subtype_df
    .drop_duplicates(['id', column])
    .groupby(['subtype', 'year'])
    .agg(
        counts=('id', 'nunique'), 
        amount=('amount', 'sum')
    )
    .reset_index()
)

tech_subtype_ts = utils.impute_empty_periods_all_ts(tech_subtype_ts, column)

utils.magnitude_and_growth(tech_subtype_ts, column, value)

,magnitude,growth,subtype
0,144718.469700,7.723290,AI
0,82759.306847,48.113274,Immersive tech
0,124198.757609,39.274742,Internet
0,326227.984343,73.912003,Mobile


In [122]:
fig = pu.ts_smooth(
    tech_subtype_ts,
    ["AI", "Immersive tech", "Internet", "Mobile"],
    variable= "amount",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

In [123]:
(165-115)/115

0.43478260869565216

## Insight 2: Applications

- Where are these technologies applied the most?
- Where do we see growth vs stagnation when it comes to applications?

In [124]:
tech_ids = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2013")
    .drop_duplicates('id')
    .id.to_list()
)

tech_ids_5y = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2019")
    .drop_duplicates('id')
    .id.to_list()
)

### Application distribution

In [131]:
column = 'type'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id', 'amount']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id', 'amount'],
    ts=True
)


In [132]:
tech_applications_df

,type,counts,counts_prop,amount,amount_prop
0,Biosciences,8,0.013,8452.022856,0.003
1,Child care & preschool,60,0.1,106088.215072,0.043
2,Development & learning,149,0.248,485392.835157,0.196
3,General,233,0.388,1389305.659891,0.561
4,Health,205,0.342,671602.014183,0.271
5,Parenting,122,0.203,293880.538124,0.119
6,Social,64,0.107,282569.01624,0.114
7,Technology,600,1.0,2474648.553527,1.0


In [133]:
fig = pu.ts_smooth(
    tech_applications_ts,
    tech_applications_ts[column].unique(),
    variable= "amount",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

In [134]:
utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='amount')

,magnitude,growth,type,counts
7,1690.404571,1748.296224,Biosciences,8
2,277861.131978,198.854162,General,233
3,134320.402837,111.304100,Health,205
5,56513.803248,80.454241,Social,64
6,494929.710705,44.777394,Technology,600
0,21217.643014,35.834434,Child care & preschool,60
1,97078.567031,30.244660,Development & learning,149
4,58776.107625,-10.111677,Parenting,122


In [531]:
# pd.set_option('display.max_colwidth', 200)
# (
#     data_exploded_df
#     .query('id in @tech_ids')
#     .query("type == 'Social'")
#     .drop_duplicates(['id'])
#     .sort_values('year', ascending=False)
# )

### Application distribution: More granular subtypes

In [135]:
column = 'subtype'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id', 'amount']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id', 'amount'],
    ts=True
)
tech_applications_df.query("type != 'Technology'").sort_values('amount', ascending=False)

,subtype,counts,counts_prop,amount,amount_prop,type
9,Infancy,153,0.255,769753.152743,0.311,General
4,Games,89,0.148,623428.932371,0.252,General
5,Health,193,0.322,615104.450156,0.249,Health
20,Parenting,122,0.203,293880.538124,0.119,Parenting
1,Cognitive development,45,0.075,239327.824641,0.097,Development & learning
12,Literacy,60,0.1,203273.878103,0.082,Development & learning
7,Inclusion,6,0.01,120438.550793,0.049,Social
19,Operations,40,0.067,88466.83447,0.036,Child care & preschool
27,Special educational needs,45,0.075,78813.491056,0.032,Development & learning
23,Prenatal,17,0.028,71447.718756,0.029,Health


In [136]:
utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='amount').sort_values(['growth'], ascending=False)

,magnitude,growth,subtype,counts,type
0,6049.334705,6369.881477,Mental health,23,Health
1,1373.811918,3930.948041,Sleep,12,Health
2,1690.404571,1748.296224,Neuroscience,8,Biosciences
3,24087.710159,1269.157450,Inclusion,6,Social
4,14289.543751,421.770242,Prenatal,17,Health
5,11235.511718,367.282849,Nutrition & weight,6,Health
6,153950.630549,282.737478,Infancy,153,General
7,124685.786474,121.899564,Games,89,General
8,9813.591223,109.881680,Labour market,11,Social
9,17693.366894,109.276115,Operations,40,Child care & preschool


In [137]:
cat_type = 'Development & learning'
cats = list(topics_df.query("type == @cat_type").subtype.unique())

In [138]:
fig = pu.ts_smooth(
    tech_applications_ts,
    cats,
    variable= "amount",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

## Insight 3: Geographical insights


In [139]:
data_countries_df = (
    data_exploded_df
    .dropna(subset=['country_code'])
    # .query("type == 'Technology'")
    # .query('subtype in @tech_subtypes')
    .drop_duplicates(['id'])    
)
country_codes = data_countries_df.country_code.unique()

growth_df = []
ts_counts = []
for country_code in country_codes:
    country_df = data_countries_df.query("country_code == @country_code")
    _ts_counts = utils.get_timeseries(country_df, column='amount').assign(country_code = country_code)
    growth_df.append(
        au.ts_magnitude_growth_(
            ts_df = _ts_counts,
            year_start = 2019,
            year_end = 2023  
        )
        .assign(country_code = country_code)
        .reset_index(drop=True)
    )
    ts_counts.append(_ts_counts)
growth_df = pd.concat(growth_df, ignore_index=True)
ts_counts = pd.concat(ts_counts, ignore_index=True)

In [140]:
(
    growth_df
    .sort_values('magnitude', ascending=False)
    .head(15)
)

,magnitude,growth,country_code
0,1.056411e+06,162.043995,USA
15,1.319021e+05,197.343121,IND
4,8.495554e+04,-68.452409,CHN
8,6.660606e+04,-66.349953,GBR
7,2.219388e+04,429.367161,CAN
3,1.286318e+04,65.319179,FRA
18,1.219857e+04,-12.464718,JPN
11,1.163632e+04,245.378124,DEU
44,1.064283e+04,1796.352833,KOR
16,8.843984e+03,208.978920,ESP


In [141]:
countries = ['GBR', 'USA', 'IND', 'CHN']
fig = pu.ts_smooth(
    ts_counts.query("country_code in @countries"),
    countries,
    variable= "amount",
    variable_title = "",
    category_column = 'country_code',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)